<a href="https://colab.research.google.com/github/jesusessu/MDD_LAB04/blob/develop/MDD_S04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Paso 1: Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import mahalanobis
from scipy.stats import zscore

In [5]:
# Paso 2: Cargar el dataset desde el repositorio UCI
url = "/content/drive/MyDrive/TECSUP/Ciclo 5/Mineria de Datos/breast-cancer-wisconsin.data"
columns = ['Sample_code_number', 'Clump_Thickness', 'Uniformity_Cell_Size',
           'Uniformity_Cell_Shape', 'Marginal_Adhesion', 'Single_Epithelial_Cell_Size',
           'Bare_Nuclei', 'Bland_Chromatin', 'Normal_Nucleoli', 'Mitoses', 'Class']
df = pd.read_csv(url, names=columns)

In [9]:
df

,Sample_code_number,Clump_Thickness,Uniformity_Cell_Size,Uniformity_Cell_Shape,Marginal_Adhesion,Single_Epithelial_Cell_Size,Bare_Nuclei,Bland_Chromatin,Normal_Nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1.0,3,1,1,0
1,1002945,5,4,4,5,7,10.0,3,2,1,0
2,1015425,3,1,1,1,2,2.0,3,1,1,0
3,1016277,6,8,8,1,3,4.0,3,7,1,0
4,1017023,4,1,1,3,2,1.0,3,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...
694,776715,3,1,1,1,3,2.0,1,1,1,0
695,841769,2,1,1,1,2,1.0,1,1,1,0
696,888820,5,10,10,3,7,3.0,8,10,2,1
697,897471,4,8,6,4,3,4.0,10,6,1,1


In [6]:
# Paso 3: Imputación de 'Bare Nuclei' y transformación de 'Class'
df.replace('?', np.nan, inplace=True)
df['Bare_Nuclei'] = pd.to_numeric(df['Bare_Nuclei'])
df['Bare_Nuclei'].fillna(df['Bare_Nuclei'].median(), inplace=True)
df['Class'] = df['Class'].replace({2: 0, 4: 1})

/tmp/ipython-input-6-1152915754.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Bare_Nuclei'].fillna(df['Bare_Nuclei'].median(), inplace=True)


In [7]:
# Paso 4: Detección y eliminación de valores atípicos univariados
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1
left = Q1 - 3 * IQR
right = Q3 + 3 * IQR
mask = ~((df < left) | (df > right)).any(axis=1)
df_uni = df[mask]

In [10]:
# Paso 5: Detección de atípicos multivariados con Mahalanobis
X = df_uni.drop(['Sample_code_number', 'Class'], axis=1)
mean = X.mean().values
cov = np.cov(X.T)
inv_cov = np.linalg.pinv(cov)

def mahalanobis_distance(row):
    return mahalanobis(row, mean, inv_cov)

distances = X.apply(mahalanobis_distance, axis=1)
df_multi = df_uni[distances < 30]

In [11]:
# Paso 6: Estadísticos descriptivos
desc_stats = df_multi.describe()

In [12]:
# Paso 7: Z-score
X_zscore = df_multi.drop(['Sample_code_number', 'Class'], axis=1)
df_zscore = X_zscore.apply(zscore)
df_zscore['Class'] = df_multi['Class'].values

/usr/local/lib/python3.11/dist-packages/pandas/core/apply.py:1081: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  results[i] = self.func(v, *self.args, **self.kwargs)


In [13]:
# Paso 8: Normalización Min-Max
df_minmax = (X_zscore - X_zscore.min()) / (X_zscore.max() - X_zscore.min())
df_minmax['Class'] = df_multi['Class'].values

# Mostrar resultados finales
print("Estadísticos descriptivos:")
print(desc_stats)

print("\nPrimeras filas con Z-score:")
print(df_zscore.head())

print("\nPrimeras filas con Min-Max:")
print(df_minmax.head())

Estadísticos descriptivos:
       Sample_code_number  Clump_Thickness  Uniformity_Cell_Size  \
count        5.770000e+02       577.000000            577.000000   
mean         1.052671e+06         3.856153              2.454073   
std          2.867939e+05         2.530049              2.590391   
min          6.163400e+04         1.000000              1.000000   
25%          8.881690e+05         2.000000              1.000000   
50%          1.173514e+06         3.000000              1.000000   
75%          1.239232e+06         5.000000              3.000000   
max          1.371920e+06        10.000000             10.000000   

       Uniformity_Cell_Shape  Marginal_Adhesion  Single_Epithelial_Cell_Size  \
count             577.000000         577.000000                   577.000000   
mean                2.580589           2.223570                     2.759099   
std                 2.554443           2.318464                     1.809271   
min                 1.000000           1